# Ocean Heating — Methods Notebook

Companion to the [dashboard](../index.html) and [`ohsi_preprocessing.py`](../ohsi_preprocessing.py).

Use this notebook to:
- Inspect pipeline outputs in `./data/`
- Sanity-check NOAA global and regional series
- Review Hobday et al. 2016 marine heatwave events before they appear in the dashboard

**Production refresh** (GitHub Actions / nightly): `python ohsi_preprocessing.py` from the repo root.

## Setup

```bash
cd ocean-heat-stress
python -m venv .venv && source .venv/bin/activate
pip install requests pandas numpy scipy matplotlib jupyter
pip install git+https://github.com/ecjoliver/marineHeatWaves.git
python ohsi_preprocessing.py   # populate ./data/ (5–10 min)
jupyter lab notebooks/methods.ipynb
```

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()   # repo root when cwd is notebooks/
DATA = ROOT / "data"

def load_json(name):
    path = DATA / name
    if not path.exists():
        print(f"Missing: {path} — run ohsi_preprocessing.py from {ROOT}")
        return None
    with open(path) as f:
        return json.load(f)

print(f"Repo root: {ROOT}")
print(f"Data dir:  {DATA} ({'exists' if DATA.is_dir() else 'missing'})")

## 1. Pipeline metadata

Same provenance record the dashboard reads for the **data-status header**.

In [ ]:
meta = load_json("meta.json")
if meta:
    for k in ("generated_utc", "ersst_status", "oisst_global_status", "ersst_years", "climatology_baseline"):
        print(f"{k}: {meta.get(k)}")
    print("regions:", meta.get("regions"))

## 2. Global ocean warming (hero chart sources)

- **ERSSTv6** via NOAAGlobalTemp v6 — native **1971–2000** baseline
- **OISST v2.1** global annual mean — same baseline on the `anom` field

In [ ]:
ersst = load_json("global_ersst.json")
oisst = load_json("global_oisst.json")

fig, ax = plt.subplots(figsize=(10, 4))

if ersst:
    df_e = pd.DataFrame(ersst)
    ax.plot(df_e["year"], df_e["anom"], color="#1A3A4A", lw=1.5, label="ERSSTv6 (NOAAGlobalTemp)")

if oisst:
    df_o = pd.DataFrame(oisst)
    ax.plot(df_o["year"], df_o["anom"], color="#E8572A", lw=1.5, label="OISST v2.1 global")

ax.axhline(0, color="#2E7DAF", ls="--", lw=0.8, alpha=0.7)
ax.set_xlabel("Year")
ax.set_ylabel("Anomaly (°C, vs 1971–2000)")
ax.set_title("Global ocean surface temperature anomaly")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Regional case studies (NOAA OISST)

Monthly SST anomalies use a **1991–2020** climatology (Hobday MHW period). Dashboard case studies: Gulf of Alaska (`blob`) and Great Barrier Reef (`gbr`).

In [ ]:
def plot_regional(name, title, color="#2E7DAF"):
    rows = load_json(f"{name}_sst.json")
    if not rows:
        return
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.fill_between(df["date"], df["anom"], 0, alpha=0.12, color=color)
    ax.plot(df["date"], df["anom"], color=color, lw=1.2)
    ax.axhline(0, color="#2E7DAF", ls="--", lw=0.8)
    ax.set_title(title)
    ax.set_ylabel("SST anomaly (°C)")
    plt.tight_layout()
    plt.show()

plot_regional("blob", "Gulf of Alaska — monthly SST anomaly")
plot_regional("gbr", "Great Barrier Reef — monthly SST anomaly", color="#6AACCA")

## 4. Marine heatwave events (Hobday et al. 2016)

Detected in `ohsi_preprocessing.py` via [`marineHeatWaves`](https://github.com/ecjoliver/marineHeatWaves).

In [ ]:
for region, label in (("blob", "Gulf of Alaska"), ("gbr", "Great Barrier Reef")):
    events = load_json(f"{region}_mhw.json")
    if not events:
        continue
    df = pd.DataFrame(events)
    print(f"\n{label}: {len(df)} events")
    display(df[["start", "end", "duration_days", "peak_intensity", "category_name"]].head(8))

## 5. Re-run the full pipeline

Uncomment to refresh all JSON from NOAA (same as GitHub Actions). Expect 5–10 minutes; ERDDAP may return **408 timeouts** — retry if needed.

In [ ]:
# import subprocess
# result = subprocess.run(["python", str(ROOT / "ohsi_preprocessing.py")], cwd=ROOT)
# print("exit code:", result.returncode)